# Overtaking 

I want to further profit from the telemetry animation and extend it to some overtaking visualizations and data insights.
1. Create a graph that shows how positions develop during a race.
2. Identify interesting overtakes. 
3. Animate overtakes (overtake replay)

## 1. Position development

https://tif1.tracinginsights.com/concepts/sessions#laps

In [55]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tif1

year = 2025
circuit = "Monaco Grand Prix"

session = tif1.get_session(year, circuit, "Race")

I need driver, lap and position (in lap):

In [56]:
positions = session.laps[["Driver", "LapNumber", "Position"]].copy()
positions.head(10)

,Driver,LapNumber,Position
0,VER,1.0,4.0
1,VER,2.0,4.0
2,VER,3.0,4.0
3,VER,4.0,4.0
4,VER,5.0,4.0
5,VER,6.0,4.0
6,VER,7.0,4.0
7,VER,8.0,4.0
8,VER,9.0,4.0
9,VER,10.0,4.0


Clean:

In [57]:
positions["LapNumber"] = pd.to_numeric(positions["LapNumber"], errors = "coerce")
positions["Position"] = pd.to_numeric(positions["Position"], errors="coerce")

positions = positions.dropna().sort_values(["Driver", "LapNumber"])

positions.head(10)

,Driver,LapNumber,Position
428,ALB,1.0,10.0
429,ALB,2.0,10.0
430,ALB,3.0,10.0
431,ALB,4.0,10.0
432,ALB,5.0,10.0
433,ALB,6.0,10.0
434,ALB,7.0,10.0
435,ALB,8.0,10.0
436,ALB,9.0,10.0
437,ALB,10.0,10.0


I need to add the starting grid / position at lap = 0 seperately: 

In [58]:
grid = session.results[["Abbreviation", "GridPosition"]].copy()

grid = grid.rename(columns={"Abbreviation": "Driver", "GridPosition": "Position"})
grid["LapNumber"] = 0

positions = pd.concat([positions, grid], ignore_index=True)
positions = positions.sort_values(["Driver", "LapNumber"])

Obtain two driver comparison from telemetry animation. Compare two drivers:

In [59]:
driver1 = "VER"
driver2 = "HAM"

fig_pos = go.Figure()

for driver in [driver1, driver2]:
    d = positions[positions["Driver"] == driver]

    fig_pos.add_trace(go.Scatter(x=d["LapNumber"], y=d["Position"], mode="lines+markers", name=driver))

fig_pos.update_yaxes(autorange="reversed")
fig_pos.update_layout(width=750, height=350, title="Race Position Development", xaxis_title="Lap", yaxis_title="Position")

fig_pos.show()

Test if it is reasonable to show all drivedrs in one graph:

In [ ]:
fig_pos = go.Figure()

for driver in positions["Driver"].unique():
    d = positions[positions["Driver"] == driver]

    fig_pos.add_trace(go.Scatter(x=d["LapNumber"], y=d["Position"], mode="lines", name=driver))

fig_pos.update_yaxes(autorange="reversed")
fig_pos.update_layout(width=900, height=550, title="Race Position Development", xaxis_title="Lap", yaxis_title="Position")

fig_pos.show()

Graph is not very readable, although user can choose which drivers he want to not look at in legend. But I want the reversed: I want to decide on which drivers I want to look at.

In [61]:
drivers = [trace.name for trace in fig_pos.data]

buttons = [dict(label="All Drivers", method="restyle", args=[{"opacity": [0.85] * len(drivers), "line.width": [2] * len(drivers)}])]

for selected in drivers:
    opacity = [1 if driver == selected else 0.5 for driver in drivers]
    width = [3 if driver == selected else 1.5 for driver in drivers]

    buttons.append(dict(label=selected, method="restyle", args=[{"opacity": opacity, "line.width": width}]))

# using AI

fig_pos.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1, xanchor="right",
        y=1.08, yanchor="bottom",
        bgcolor="white",
        bordercolor="lightgray",
        showactive=True
    )],
    margin=dict(t=110, r=140),
    hovermode="closest"
)

fig_pos.show()

Maybe a slider would enhance the plot too:

In [ ]:
drivers = [trace.name for trace in fig_pos.data]
laps = sorted(positions["LapNumber"].astype(int).unique())

# vertical line showing the selected lap
fig_pos.add_vline(x=laps[0], line_dash="dash", line_color="black", line_width=2)

# lap slider
steps = []

for lap in laps:
    steps.append(dict(
        method="relayout",
        args=[{"shapes[0].x0": int(lap), "shapes[0].x1": int(lap)}],
        label=str(lap)
    ))

fig_pos.update_layout(sliders=[dict(
    active=0,
    steps=steps,
    currentvalue={"prefix": "Lap: "},
    x=0.05, y=-0.15, len=0.9
)])

# driver selection
buttons = [dict(
    label="All Drivers",
    method="restyle",
    args=[{"opacity": [0.85] * len(drivers), "line.width": [2] * len(drivers)}]
)]

# using AI

for selected in drivers:
    opacity = [1 if driver == selected else 0.4 for driver in drivers]
    width = [4 if driver == selected else 1.5 for driver in drivers]

    buttons.append(dict(
        label=selected,
        method="restyle",
        args=[{"opacity": opacity, "line.width": width}]
    ))

fig_pos.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1, xanchor="right",
        y=1.08, yanchor="bottom",
        bgcolor="white",
        bordercolor="lightgray",
        showactive=True
    )],
    margin=dict(t=110, r=140, b=120),
    hovermode="closest"
)

fig_pos.show()

(Instead of selecting the driver through a dropdown, I would like to select him with clicking on the line in the graph. I used AI to code it, but testing it showed performance problems. Therefore I also deleted it from the notebook. AI suggests a browser-side solution. I omit it for now.)

Turn it into a function and add design using AI. The driver selection is done in streamlit rather than directly in the graph, therefore I introduce an additional function parameter.

In [89]:
# F1 DESIGN
BACKGROUND = "#101014"
GRID = "#303038"
WHITE = "#FFFFFF"
MUTED = "#AAAAAA"
GREEN = "#19B99B"
RED = "#E5484D"
BLUE = "#5695E8"


def f1_style(fig, title, subtitle, height=600):
    # Apply consistent F1 styling to all plots.

    fig.update_layout(
        title=dict(
            text=(
                f"<b>F1 / {title}</b>"
                f"<br><span style='font-size:14px;color:{MUTED}'>"
                f"{subtitle}</span>"
            ),
            font=dict(size=23, color=WHITE),
            x=0.04,
            y=0.97,
            xanchor="left",
            yanchor="top"
        ),
        template="plotly_dark",
        paper_bgcolor=BACKGROUND,
        plot_bgcolor=BACKGROUND,
        font=dict(family="Arial", color=WHITE, size=13),
        height=height,
        margin=dict(l=180, r=70, t=120, b=70),
        showlegend=False
    )

    fig.update_xaxes(
        gridcolor=GRID,
        zeroline=False,
        tickfont=dict(color=MUTED)
    )

    fig.update_yaxes(
        showgrid=False,
        tickfont=dict(color=WHITE, size=13)
    )

    return fig

In [92]:
def plot_position_development(year, circuit, selected_driver=None):

    session = tif1.get_session(year, circuit, "Race")

    # prepare lap positions
    positions = session.laps[["Driver", "LapNumber", "Position"]].copy()
    positions = positions.dropna()

    # add starting grid
    grid = session.results[["Abbreviation", "GridPosition"]].copy()
    grid = grid.rename(columns={"Abbreviation": "Driver", "GridPosition": "Position"})
    grid["LapNumber"] = 0

    positions = pd.concat([positions, grid], ignore_index=True)
    positions = positions[positions["Position"] > 0]
    positions = positions.sort_values(["Driver", "LapNumber"])

    drivers = sorted(positions["Driver"].unique())
    laps = sorted(positions["LapNumber"].astype(int).unique())

    # plot driver trajectories
    fig = go.Figure()

    for driver in drivers:
        d = positions[positions["Driver"] == driver]

        highlighted = selected_driver is None or driver == selected_driver

        fig.add_trace(go.Scatter(
            x=d["LapNumber"],
            y=d["Position"],
            mode="lines",
            name=driver,
            opacity=0.85 if selected_driver is None else (1 if highlighted else 0.4),
            line=dict(width=4 if selected_driver == driver else 2),
            hovertemplate=f"<b>{driver}</b><br>Lap %{{x}}<br>Position P%{{y}}<extra></extra>"
        ))

    # apply F1 styling
    fig = f1_style(
        fig,
        title="RACE POSITION DEVELOPMENT",
        subtitle=f"{year} {circuit} | Lap-by-Lap Race Positions",
        height=650
    )

    fig.update_layout(
        margin=dict(l=70, r=50, t=125, b=125),
        hovermode="closest"
    )

    fig.update_xaxes(title="Lap", dtick=5)
    fig.update_yaxes(
        title="Position",
        autorange="reversed",
        dtick=1,
        showgrid=True,
        gridcolor=GRID
    )

    # lap slider
    fig.add_vline(x=laps[0], line_dash="dash", line_color=WHITE, opacity=0.6)

    steps = []

    for lap in laps:
        steps.append(dict(
            method="relayout",
            args=[{"shapes[0].x0": int(lap), "shapes[0].x1": int(lap)}],
            label=str(lap)
        ))

    fig.update_layout(sliders=[dict(
        active=0,
        steps=steps,
        currentvalue={"prefix": "Lap: ", "font": {"color": WHITE}},
        x=0.05, y=-0.12, len=0.9
    )])

    return fig

In [93]:
plot_position_development(2025, "Monaco Grand Prix")

## 2. Identify interesting overtakes:

Calculate position changes:

In [63]:
positions["PreviousPosition"] = positions.groupby("Driver")["Position"].shift(1)
positions["PreviousLap"] = positions.groupby("Driver")["LapNumber"].shift(1)

positions["Change"] = positions["PreviousPosition"] - positions["Position"]

changes = positions[positions["LapNumber"] == positions["PreviousLap"] + 1].copy()

changes[["Driver", "LapNumber", "PreviousPosition", "Position", "Change"]].head(10)

,Driver,LapNumber,PreviousPosition,Position,Change
0,ALB,1.0,10.0,10.0,0.0
1,ALB,2.0,10.0,10.0,0.0
2,ALB,3.0,10.0,10.0,0.0
3,ALB,4.0,10.0,10.0,0.0
4,ALB,5.0,10.0,10.0,0.0
5,ALB,6.0,10.0,10.0,0.0
6,ALB,7.0,10.0,10.0,0.0
7,ALB,8.0,10.0,10.0,0.0
8,ALB,9.0,10.0,10.0,0.0
9,ALB,10.0,10.0,10.0,0.0


Find the largest position changes:

In [64]:
interesting = changes[changes["Change"].abs() >= 2]

interesting.sort_values("Change", ascending=False)[["Driver", "LapNumber", "PreviousPosition", "Position", "Change"]]

,Driver,LapNumber,PreviousPosition,Position,Change
339,COL,1.0,18.0,15.0,3.0
1194,STR,1.0,19.0,16.0,3.0
279,BOR,17.0,17.0,15.0,2.0
212,BEA,26.0,19.0,17.0,2.0
594,HUL,17.0,18.0,16.0,2.0
1219,STR,26.0,18.0,16.0,2.0
355,COL,17.0,19.0,17.0,2.0
441,HAD,19.0,6.0,8.0,-2.0
684,LAW,31.0,8.0,10.0,-2.0
364,COL,26.0,17.0,19.0,-2.0


More interesting / more relevant for the two drivers animation would be to find two drivers that switch positions.

I 

In [65]:
# make sure no negative positions are inside the data (rigorous testing for later (maybe messy) data over all years)
pos = positions[positions["Position"] > 0].pivot_table(index="LapNumber", columns="Driver", values="Position") 

events = []

# do not iterate over first lap [1:]
for lap in pos.index[1:]:
    # lap should exist (rigorous testing for later messy data)
    if lap - 1 not in pos.index:
        continue

    before = pos.loc[lap - 1]
    after = pos.loc[lap]

    # compare every pair of drivers
    for i, a in enumerate(pos.columns):
        # only selecting drivers in front of driver a (no duplicate comparisons)
        for b in pos.columns[i + 1:]:
            # skip missing observations (rigorous testing for later messy data)
            if pd.isna(before[a]) or pd.isna(before[b]) or pd.isna(after[a]) or pd.isna(after[b]):
                continue
            # detect position reversals
            if before[a] > before[b] and after[a] < after[b]:
                events.append([lap, a, b])

            elif before[b] > before[a] and after[b] < after[a]:
                events.append([lap, b, a])

overtakes = pd.DataFrame(events, columns=["LapNumber", "GainedAhead", "FellBehind"])

overtakes.head(10)

,LapNumber,GainedAhead,FellBehind
0,1.0,ANT,TSU
1,1.0,BEA,BOR
2,1.0,COL,BOR
3,1.0,GAS,BOR
4,1.0,STR,BOR
5,1.0,COL,GAS
6,1.0,COL,TSU
7,1.0,STR,GAS
8,1.0,HUL,TSU
9,1.0,RUS,TSU


I should differentiate position changes due to pit stops: 

In [66]:
pit = session.laps["PitInTime"].notna() | session.laps["PitOutTime"].notna()

pit_laps = session.laps[pit]

pit_set = set(zip(pit_laps["Driver"], pit_laps["LapNumber"]))

overtakes["PitEvent"] = [(a, lap) in pit_set or (b, lap) in pit_set for lap, a, b in overtakes.itertuples(index=False, name=None)]

overtakes[~overtakes["PitEvent"]]

,LapNumber,GainedAhead,FellBehind,PitEvent
62,42.0,SAI,ALB,False
67,48.0,RUS,ALB,False
72,64.0,ALB,SAI,False
88,76.0,STR,HUL,False


# 3. Overtake Replay

Select a candidate:

In [67]:
candidates = overtakes[~overtakes["PitEvent"]].reset_index(drop=True)

candidates.head(10)

,LapNumber,GainedAhead,FellBehind,PitEvent
0,42.0,SAI,ALB,False
1,48.0,RUS,ALB,False
2,64.0,ALB,SAI,False
3,76.0,STR,HUL,False


In [69]:
event = candidates.iloc[0]

lap = int(event["LapNumber"])
driver1 = event["GainedAhead"]
driver2 = event["FellBehind"]

print(f"Lap: {lap}: {driver1} moves ahead of {driver2}!")

Lap: 42: SAI moves ahead of ALB!


I use the function from the telemetry_animation and add lap number as a parameter. Adjusting the drivers to the same time grid is not used for overtakes. I used AI to clean the function a bit.

In [ ]:
def telemetry_viz_2drivers_messages_subplot(year, circuit, driver1, driver2, fps=20, lap_number=None):
    # F1 styling
    BACKGROUND = "#101014"
    GRID = "#303038"
    WHITE = "#FFFFFF"
    MUTED = "#AAAAAA"
    RED = "#E5484D"
    BLUE = "#36A3FF"

    # 1. Load circuit and telemetry
    session = tif1.get_session(year, circuit, "Race")
    track = session.get_circuit_info().track
    drivers = [driver1, driver2]
    colors = [RED, BLUE]
    telemetry_data = {}

    # column name differs for fastest lap vs. overtake animation
    time_col = "Time" if lap_number is None else "SessionTime"

    for driver in drivers:
        if lap_number is None:
            telemetry = session.get_driver(driver).get_fastest_lap_tel()
        else:
            # for the overtake the telemetry of the specific lap is imported from tif1
            telemetry = session.get_driver(driver).get_lap(lap_number).telemetry

        telemetry = telemetry.dropna(subset=[time_col, "X", "Y"]).sort_values(time_col).drop_duplicates(time_col).copy()
        if len(telemetry) < 2:
            raise ValueError(f"Not enough telemetry for {driver} on the requested lap.")

        time = telemetry[time_col].dt.total_seconds().to_numpy()
        # adjusting drivers to the same time grid is not used for overtakes
        if lap_number is None:
            time = time - time[0]

        telemetry["TimeSeconds"] = time
        telemetry_data[driver] = {"telemetry": telemetry, "time": time}

    # 2. Shared time axis only for fastest lap animation
    if lap_number is None:
        start = 0
        end = max(telemetry_data[d]["time"][-1] for d in drivers)
    else:
        # the animation starts when the driver which is behind is at time zero of the lap
        start = max(telemetry_data[d]["time"][0] for d in drivers)
        # the animation ends when the driver which is ahead is at the last time of the lap
        end = min(telemetry_data[d]["time"][-1] for d in drivers)

    if end <= start:
        raise ValueError("The selected laps do not overlap in race time. Try another event.")

    time_interp = np.arange(start, end, 1 / fps)

    for driver in drivers:
        data = telemetry_data[driver]
        telemetry = data["telemetry"]
        data["x"] = np.interp(time_interp, data["time"], telemetry["X"])
        data["y"] = np.interp(time_interp, data["time"], telemetry["Y"])

    # 3. Section messages only for fastest-lap animation
    if lap_number is None:
        corners = session.get_circuit_info().corners.sort_values("Distance")
        reference = telemetry_data[driver1]["telemetry"][ ["Distance", "RelativeDistance"] ].dropna()
        reference = reference[reference["RelativeDistance"] > 0.01]
        lap_distance = np.median(reference["Distance"] / reference["RelativeDistance"])

        section_bounds = np.concatenate(([0], corners["Distance"].to_numpy(), [lap_distance]))
        n_sections = len(section_bounds) - 1
        section_names = (["Start → T1"] + [f"T{i} → T{i+1}" for i in range(1, n_sections - 1)] + [f"T{n_sections - 1} → Finish"])
        sections = pd.DataFrame({"Section": section_names})

        def corner_times(telemetry):
            times = []
            for _, corner in corners.iterrows():
                expected = corner["Distance"] / lap_distance
                nearby = telemetry[telemetry["RelativeDistance"].between(expected - 0.01, expected + 0.01)]
                if nearby.empty:
                    raise ValueError("A corner could not be matched to telemetry.")
                distances = (nearby["X"] - corner["X"])**2 + (nearby["Y"] - corner["Y"])**2
                times.append(nearby.loc[distances.idxmin(), "TimeSeconds"])
            return np.array(times)

        t1 = telemetry_data[driver1]["telemetry"]
        t2 = telemetry_data[driver2]["telemetry"]
        time1 = np.concatenate(([0], corner_times(t1), [t1["TimeSeconds"].max()]))
        time2 = np.concatenate(([0], corner_times(t2), [t2["TimeSeconds"].max()]))

        sections["Delta"] = np.diff(time1) - np.diff(time2)
        sections["Winner"] = np.where(sections["Delta"] > 0, driver2, driver1)
        sections["Message"] = (sections["Winner"] + " gained " + sections["Delta"].abs().round(3).astype(str) + "s in " + sections["Section"])
        message_times = np.maximum(time1[1:], time2[1:])
        section_x = np.concatenate((corners["X"].to_numpy(), [track["X"].iloc[0]]))
        section_y = np.concatenate((corners["Y"].to_numpy(), [track["Y"].iloc[0]]))

    # 4. Plot the circuit and cars
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=track["X"], y=track["Y"], mode="lines", name="Circuit",
        line=dict(color="#666674", width=5), hoverinfo="skip"
    ))

    for driver, marker in zip(drivers, [dict(symbol="diamond", color=RED, size=14),
                                        dict(symbol="circle", color=BLUE, size=11)]):
        fig.add_trace(go.Scatter(
            x=[telemetry_data[driver]["x"][0]], y=[telemetry_data[driver]["y"][0]],
            mode="markers", name=driver, marker=marker, hoverinfo="skip"
        ))

    # figure title depends on "mode"
    mode = "Fastest Lap" if lap_number is None else f"Lap {lap_number} / Race Replay"
    fig.update_layout(
        title=dict(
            text=f"<b>F1 / LAP TELEMETRY</b><br><span style='font-size:14px;color:{MUTED}'>{year} {circuit} | <span style='color:{RED}'>{driver1}</span> vs <span style='color:{BLUE}'>{driver2}</span> | {mode}</span>",
            font=dict(size=23, color=WHITE), x=0.04, y=0.97, xanchor="left", yanchor="top"
        ),
        template="plotly_dark", paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
        font=dict(family="Arial", color=WHITE, size=13),
        width=750, height=670, margin=dict(l=70, r=40, t=110, b=95),
        showlegend=False, hovermode=False
    )

    x_pad = (track["X"].max() - track["X"].min()) * 0.30
    y_pad = (track["Y"].max() - track["Y"].min()) * 0.30
    fig.update_xaxes(visible=False, range=[track["X"].min() - x_pad, track["X"].max() + x_pad])
    fig.update_yaxes(visible=False, range=[track["Y"].min() - y_pad, track["Y"].max() + y_pad], scaleanchor="x", scaleratio=1)

    # 5. Prepare section annotations once (fastest-lap mode only)
    section_annotations = []
    if lap_number is None:
        for j in range(len(sections)):
            distance = (track["X"] - section_x[j])**2 + (track["Y"] - section_y[j])**2
            k = distance.argmin()
            before, after = (k - 1) % len(track), (k + 1) % len(track)
            dx = track["X"].iloc[after] - track["X"].iloc[before]
            dy = track["Y"].iloc[after] - track["Y"].iloc[before]
            length = np.sqrt(dx**2 + dy**2)
            ax, ay = (-dy / length * 150, -dx / length * 150) if length > 0 else (120, -90)
            color = RED if sections["Winner"].iloc[j] == driver1 else BLUE
            text = f"<span style='font-size:10px;color:{MUTED}'>SECTION {j+1:02d}</span><br><b>{sections['Message'].iloc[j]}</b>"
            section_annotations.append(dict(
                x=section_x[j], y=section_y[j], text=text, showarrow=True,
                arrowhead=2, arrowwidth=2, arrowcolor=color, ax=ax, ay=ay,
                bgcolor="#202027", bordercolor=color, borderwidth=1.5,
                borderpad=8, font=dict(size=12, color=WHITE)
            ))

    # 6. Animation
    frames = []
    for i, t in enumerate(time_interp):
        positions = [go.Scatter(x=[telemetry_data[d]["x"][i]], y=[telemetry_data[d]["y"][i]]) for d in drivers]
        annotations = []
        if lap_number is None:
            completed = np.searchsorted(message_times, t, side="right")
            if completed > 0:
                annotations = [section_annotations[completed - 1]]

        frames.append(go.Frame(data=positions, traces=[1, 2], layout=go.Layout(annotations=annotations)))

    fig.frames = frames
    fig.update_layout(updatemenus=[dict(
        type="buttons", direction="left", showactive=False,
        x=0.5, xanchor="center", y=-0.08, yanchor="top",
        bgcolor="#202027", bordercolor=GRID, font=dict(color=WHITE, size=13),
        buttons=[
            dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 1000 / fps, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}]),
            dict(label="Ⅱ Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}, "mode": "immediate"}])
        ]
    )])
    return fig


An overtake replay is then called like this:

In [72]:
event = candidates.iloc[0]

lap = int(event["LapNumber"])
driver1 = event["GainedAhead"]
driver2 = event["FellBehind"]

fig = telemetry_viz_2drivers_messages_subplot(
    year, circuit, driver1, driver2,
    lap_number=lap,
    fps=20
)

fig.show()

The animation of the whole lap takes too long. I want to only show +-10 seconds around the overtake. 

In [75]:
event = candidates.iloc[0]

lap = int(event["LapNumber"])
driver1 = event["GainedAhead"]
driver2 = event["FellBehind"]

telemetry1 = session.get_driver(driver1).get_lap(lap).telemetry
telemetry2 = session.get_driver(driver2).get_lap(lap).telemetry

telemetry1[["SessionTime", "RelativeDistance"]].head()

,SessionTime,RelativeDistance
0,0 days 01:52:00.512000,0.000009
1,0 days 01:52:00.590000,0.001750
2,0 days 01:52:00.639000,0.002850
3,0 days 01:52:00.771000,0.005838
4,0 days 01:52:00.879000,0.008301


Identify overtaking time:

In [79]:
t1 = telemetry1["SessionTime"].dt.total_seconds().to_numpy()
t2 = telemetry2["SessionTime"].dt.total_seconds().to_numpy()

start = max(t1.min(), t2.min())
end = min(t1.max(), t2.max())

if end <= start:
    raise ValueError("No overlapping telemetry")

time = np.arange(start, end, 0.1)

p1 = np.interp(time, t1, telemetry1["RelativeDistance"])
p2 = np.interp(time, t2, telemetry2["RelativeDistance"])

difference = p1 - p2

difference

array([-2.27065838e-02, -2.27119017e-02, -2.26520491e-02, -2.25547900e-02,
       -2.24071131e-02, -2.21415814e-02, -2.18564029e-02, -2.13239737e-02,
       -2.06768630e-02, -2.00040692e-02, -1.92935210e-02, -1.85250294e-02,
       -1.77232644e-02, -1.68999100e-02, -1.60586487e-02, -1.52217103e-02,
       -1.44007300e-02, -1.36402064e-02, -1.30264051e-02, -1.24621053e-02,
       -1.19823927e-02, -1.15267929e-02, -1.10965079e-02, -1.06820395e-02,
       -1.04296158e-02, -1.02547946e-02, -1.00799733e-02, -9.91328614e-03,
       -9.76236362e-03, -9.64893967e-03, -9.55785668e-03, -9.47269005e-03,
       -9.40142562e-03, -9.42541115e-03, -9.51628352e-03, -9.60715589e-03,
       -9.69848856e-03, -9.85278278e-03, -1.00088602e-02, -1.02181537e-02,
       -1.04694900e-02, -1.07296090e-02, -1.09961772e-02, -1.12635097e-02,
       -1.15682903e-02, -1.19198783e-02, -1.22749894e-02, -1.26379798e-02,
       -1.30011975e-02, -1.33605435e-02, -1.37141973e-02, -1.40294697e-02,
       -1.43420837e-02, -

In [81]:

if difference[0] < 0 and (difference >= 0).any():
    overtake_time = time[np.where(difference >= 0)[0][0]]
else:
    print("No overtake detected")

print("Approximate overtake time:", overtake_time)

Approximate overtake time: 6772.81200000019


Add it to the function: 

In [ ]:
def telemetry_viz_2drivers_messages_subplot(year, circuit, driver1, driver2, fps=20, lap_number=None):
    # F1 styling
    BACKGROUND = "#101014"
    GRID = "#303038"
    WHITE = "#FFFFFF"
    MUTED = "#AAAAAA"
    RED = "#E5484D"
    BLUE = "#36A3FF"

    # 1. Load circuit and telemetry
    session = tif1.get_session(year, circuit, "Race")
    track = session.get_circuit_info().track
    drivers = [driver1, driver2]
    colors = [RED, BLUE]
    telemetry_data = {}

    # column name differs for fastest lap vs. overtake animation
    time_col = "Time" if lap_number is None else "SessionTime"

    for driver in drivers:
        if lap_number is None:
            telemetry = session.get_driver(driver).get_fastest_lap_tel()
        else:
            # for the overtake the telemetry of the specific lap is imported from tif1
            telemetry = session.get_driver(driver).get_lap(lap_number).telemetry

        telemetry = telemetry.dropna(subset=[time_col, "X", "Y"]).sort_values(time_col).drop_duplicates(time_col).copy()
        if len(telemetry) < 2:
            raise ValueError(f"Not enough telemetry for {driver} on the requested lap.")

        time = telemetry[time_col].dt.total_seconds().to_numpy()
        # adjusting drivers to the same time grid is not used for overtakes
        if lap_number is None:
            time = time - time[0]

        telemetry["TimeSeconds"] = time
        telemetry_data[driver] = {"telemetry": telemetry, "time": time}

    # 2. Shared time axis only for fastest lap animation
    if lap_number is None:
        start = 0
        end = max(telemetry_data[d]["time"][-1] for d in drivers)
    else:
        # find the time interval where both drivers have telemetry
        start = max(telemetry_data[d]["time"][0] for d in drivers)
        end = min(telemetry_data[d]["time"][-1] for d in drivers)

    if end <= start:
        raise ValueError("The selected laps do not overlap in race time.")

    # NEW: Find overtaking time
    if lap_number is not None:

        time_check = np.arange(start, end, 0.1)

        d1 = telemetry_data[driver1]
        d2 = telemetry_data[driver2]

        p1 = np.interp(time_check, d1["time"], d1["telemetry"]["RelativeDistance"])
        p2 = np.interp(time_check, d2["time"], d2["telemetry"]["RelativeDistance"])

        difference = p1 - p2

        if difference[0] < 0 and (difference >= 0).any():
            overtake_time = time_check[np.where(difference >= 0)[0][0]]

            start = max(start, overtake_time - 10)
            end = min(end, overtake_time + 10)

        else:
            print("No overtake detected. Showing the full lap.")

    time_interp = np.arange(start, end, 1 / fps)

    for driver in drivers:
        data = telemetry_data[driver]
        telemetry = data["telemetry"]
        data["x"] = np.interp(time_interp, data["time"], telemetry["X"])
        data["y"] = np.interp(time_interp, data["time"], telemetry["Y"])

    # 3. Section messages only for fastest-lap animation
    if lap_number is None:
        corners = session.get_circuit_info().corners.sort_values("Distance")
        reference = telemetry_data[driver1]["telemetry"][ ["Distance", "RelativeDistance"] ].dropna()
        reference = reference[reference["RelativeDistance"] > 0.01]
        lap_distance = np.median(reference["Distance"] / reference["RelativeDistance"])

        section_bounds = np.concatenate(([0], corners["Distance"].to_numpy(), [lap_distance]))
        n_sections = len(section_bounds) - 1
        section_names = (["Start → T1"] + [f"T{i} → T{i+1}" for i in range(1, n_sections - 1)] + [f"T{n_sections - 1} → Finish"])
        sections = pd.DataFrame({"Section": section_names})

        def corner_times(telemetry):
            times = []
            for _, corner in corners.iterrows():
                expected = corner["Distance"] / lap_distance
                nearby = telemetry[telemetry["RelativeDistance"].between(expected - 0.01, expected + 0.01)]
                if nearby.empty:
                    raise ValueError("A corner could not be matched to telemetry.")
                distances = (nearby["X"] - corner["X"])**2 + (nearby["Y"] - corner["Y"])**2
                times.append(nearby.loc[distances.idxmin(), "TimeSeconds"])
            return np.array(times)

        t1 = telemetry_data[driver1]["telemetry"]
        t2 = telemetry_data[driver2]["telemetry"]
        time1 = np.concatenate(([0], corner_times(t1), [t1["TimeSeconds"].max()]))
        time2 = np.concatenate(([0], corner_times(t2), [t2["TimeSeconds"].max()]))

        sections["Delta"] = np.diff(time1) - np.diff(time2)
        sections["Winner"] = np.where(sections["Delta"] > 0, driver2, driver1)
        sections["Message"] = (sections["Winner"] + " gained " + sections["Delta"].abs().round(3).astype(str) + "s in " + sections["Section"])
        message_times = np.maximum(time1[1:], time2[1:])
        section_x = np.concatenate((corners["X"].to_numpy(), [track["X"].iloc[0]]))
        section_y = np.concatenate((corners["Y"].to_numpy(), [track["Y"].iloc[0]]))

    # 4. Plot the circuit and cars
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=track["X"], y=track["Y"], mode="lines", name="Circuit",
        line=dict(color="#666674", width=5), hoverinfo="skip"
    ))

    for driver, marker in zip(drivers, [dict(symbol="diamond", color=RED, size=14),
                                        dict(symbol="circle", color=BLUE, size=11)]):
        fig.add_trace(go.Scatter(
            x=[telemetry_data[driver]["x"][0]], y=[telemetry_data[driver]["y"][0]],
            mode="markers", name=driver, marker=marker, hoverinfo="skip"
        ))

    # figure title depends on "mode"
    mode = "Fastest Lap" if lap_number is None else f"Lap {lap_number} / Race Replay"
    fig.update_layout(
        title=dict(
            text=f"<b>F1 / LAP TELEMETRY</b><br><span style='font-size:14px;color:{MUTED}'>{year} {circuit} | <span style='color:{RED}'>{driver1}</span> vs <span style='color:{BLUE}'>{driver2}</span> | {mode}</span>",
            font=dict(size=23, color=WHITE), x=0.04, y=0.97, xanchor="left", yanchor="top"
        ),
        template="plotly_dark", paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
        font=dict(family="Arial", color=WHITE, size=13),
        width=750, height=670, margin=dict(l=70, r=40, t=110, b=95),
        showlegend=False, hovermode=False
    )

    x_pad = (track["X"].max() - track["X"].min()) * 0.30
    y_pad = (track["Y"].max() - track["Y"].min()) * 0.30
    fig.update_xaxes(visible=False, range=[track["X"].min() - x_pad, track["X"].max() + x_pad])
    fig.update_yaxes(visible=False, range=[track["Y"].min() - y_pad, track["Y"].max() + y_pad], scaleanchor="x", scaleratio=1)

    # 5. Prepare section annotations once (fastest-lap mode only)
    section_annotations = []
    if lap_number is None:
        for j in range(len(sections)):
            distance = (track["X"] - section_x[j])**2 + (track["Y"] - section_y[j])**2
            k = distance.argmin()
            before, after = (k - 1) % len(track), (k + 1) % len(track)
            dx = track["X"].iloc[after] - track["X"].iloc[before]
            dy = track["Y"].iloc[after] - track["Y"].iloc[before]
            length = np.sqrt(dx**2 + dy**2)
            ax, ay = (-dy / length * 150, -dx / length * 150) if length > 0 else (120, -90)
            color = RED if sections["Winner"].iloc[j] == driver1 else BLUE
            text = f"<span style='font-size:10px;color:{MUTED}'>SECTION {j+1:02d}</span><br><b>{sections['Message'].iloc[j]}</b>"
            section_annotations.append(dict(
                x=section_x[j], y=section_y[j], text=text, showarrow=True,
                arrowhead=2, arrowwidth=2, arrowcolor=color, ax=ax, ay=ay,
                bgcolor="#202027", bordercolor=color, borderwidth=1.5,
                borderpad=8, font=dict(size=12, color=WHITE)
            ))

    # 6. Animation
    frames = []
    for i, t in enumerate(time_interp):
        positions = [go.Scatter(x=[telemetry_data[d]["x"][i]], y=[telemetry_data[d]["y"][i]]) for d in drivers]
        annotations = []
        if lap_number is None:
            completed = np.searchsorted(message_times, t, side="right")
            if completed > 0:
                annotations = [section_annotations[completed - 1]]

        frames.append(go.Frame(data=positions, traces=[1, 2], layout=go.Layout(annotations=annotations)))

    fig.frames = frames
    fig.update_layout(updatemenus=[dict(
        type="buttons", direction="left", showactive=False,
        x=0.5, xanchor="center", y=-0.08, yanchor="top",
        bgcolor="#202027", bordercolor=GRID, font=dict(color=WHITE, size=13),
        buttons=[
            dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 1000 / fps, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}]),
            dict(label="Ⅱ Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}, "mode": "immediate"}])
        ]
    )])
    return fig


In [83]:
event = candidates.iloc[0]

lap = int(event["LapNumber"])
driver1 = event["GainedAhead"]
driver2 = event["FellBehind"]

fig = telemetry_viz_2drivers_messages_subplot(
    year, circuit, driver1, driver2,
    lap_number=lap,
    fps=20
)

fig.show()

### Create a function to get overtake candidates:

Here I put identify candidates code into a function. Additionally, I add a limit of 8 highlights (AI coded) and include nice headlines. Putting together the function was assisted by AI. 

In [ ]:
def get_overtake_highlights(year, circuit):

    session = tif1.get_session(year, circuit, "Race")
    laps = session.laps.copy()

    # Prepare position data
    positions = laps[["Driver", "LapNumber", "Position"]].copy()
    positions = positions.dropna().sort_values(["Driver", "LapNumber"])

    # Add starting grid as lap 0
    grid = session.results[["Abbreviation", "GridPosition"]].copy()

    grid = grid.rename(columns={
        "Abbreviation": "Driver",
        "GridPosition": "Position"
    })

    grid["LapNumber"] = 0

    positions = pd.concat([positions, grid], ignore_index=True)
    positions = positions[positions["Position"] > 0]
    positions = positions.sort_values(["Driver", "LapNumber"])

    # Position table
    pos = positions.pivot_table(
        index="LapNumber", columns="Driver", values="Position"
    )

    # Identify pit-stop laps
    pit = laps["PitInTime"].notna() | laps["PitOutTime"].notna()
    pit_laps = laps[pit]
    pit_set = set(zip(pit_laps["Driver"], pit_laps["LapNumber"]))

    # Identify potential overtakes
    events = []

    for lap in pos.index[1:]:
        if lap - 1 not in pos.index:
            continue

        before = pos.loc[lap - 1]
        after = pos.loc[lap]

        for i, a in enumerate(pos.columns):
            for b in pos.columns[i + 1:]:

                if pd.isna(before[a]) or pd.isna(before[b]) or pd.isna(after[a]) or pd.isna(after[b]):
                    continue

                if before[a] > before[b] and after[a] < after[b]:
                    winner, loser = a, b

                elif before[b] > before[a] and after[b] < after[a]:
                    winner, loser = b, a

                else:
                    continue

                if (winner, lap) in pit_set or (loser, lap) in pit_set:
                    continue

                events.append({"LapNumber": int(lap), "GainedAhead": winner, "FellBehind": loser, "NewPosition": int(after[winner]), "PositionsGained": int(before[winner] - after[winner])})

    # Select interesting overtakes
    highlights = pd.DataFrame(events, columns=[
        "LapNumber", "GainedAhead", "FellBehind",
        "NewPosition", "PositionsGained"
    ])

    last_lap = pos.index.max()

    highlights["Category"] = np.select(
        [
            highlights["NewPosition"] == 1,
            highlights["NewPosition"] <= 3,
            (highlights["LapNumber"] >= 0.75 * last_lap) & (highlights["NewPosition"] <= 10),
            (highlights["PositionsGained"] >= 2) & (highlights["NewPosition"] <= 10)
        ],
        ["Lead change", "Podium battle", "Late-race battle", "Big position gain"],
        default="Other"
    )

    highlights = highlights[
        (highlights["Category"] != "Other") &
        (highlights["LapNumber"] > 1)
    ].copy()

    highlights = highlights.sort_values(["LapNumber", "NewPosition"])

    highlights["Headline"] = (
        highlights["Category"] + "! " +
        highlights["GainedAhead"] + " moves ahead of " +
        highlights["FellBehind"]
    )

    return highlights.reset_index(drop=True)

In [85]:
import tif1

session = tif1.get_session(2025, "Monaco Grand Prix", "Race")

display(session.drivers_df.head())
display(session.laps.head())

,Driver,Team,DriverNumber,FirstName,LastName,TeamColor,HeadshotUrl
0,NOR,McLaren,4,Lando,Norris,F47600,https://media.formula1.com/d_driver_fallback_i...
1,LEC,Ferrari,16,Charles,Leclerc,ED1131,https://media.formula1.com/d_driver_fallback_i...
2,PIA,McLaren,81,Oscar,Piastri,F47600,https://media.formula1.com/d_driver_fallback_i...
3,VER,Red Bull Racing,1,Max,Verstappen,4781D7,https://media.formula1.com/d_driver_fallback_i...
4,HAM,Ferrari,44,Lewis,Hamilton,ED1131,https://media.formula1.com/d_driver_fallback_i...


,index,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,...,Rainfall,TrackTemp,WindDirection,WindSpeed,LapTimeSeconds,ms1,ms2,ms3,drv,team
0,0,0 days 00:57:36.090000,VER,1,0 days 00:01:27.020000,1.0,1.0,NaT,NaT,NaT,...,False,45.2,345,0.8,87.020,111111,1111111113,11111,VER,Red Bull Racing
1,1,0 days 00:59:22.408000,VER,1,0 days 00:01:46.318000,2.0,1.0,NaT,NaT,0 days 00:00:34.949000,...,False,42.7,331,0.9,106.318,310000,0000000000,00000,VER,Red Bull Racing
2,2,0 days 01:01:11.223000,VER,1,0 days 00:01:48.815000,3.0,1.0,NaT,NaT,0 days 00:00:32.110000,...,False,43.2,346,1.6,108.815,000000,0000010000,00000,VER,Red Bull Racing
3,3,0 days 01:02:51.513000,VER,1,0 days 00:01:40.290000,4.0,1.0,NaT,NaT,0 days 00:00:31.006000,...,False,43.2,0,1.0,100.290,700000,0000000000,10111,VER,Red Bull Racing
4,4,0 days 01:04:12.313000,VER,1,0 days 00:01:20.800000,5.0,1.0,NaT,NaT,0 days 00:00:21.744000,...,False,41.3,318,0.5,80.800,711000,1113111000,00100,VER,Red Bull Racing
